# Assignment 6: Data Scraping
```
Course: ENV 700 - Environmental Data Exploration
Author: Student Name
Date: Fall 2026
```

This notebook accompanies the Python lessons on **Data Scraping**. We'll be scraping data from the North Carolina Department of Water Quality (NC DWQ). 

## 1. Setup
* Import packages: `requests`, `BeautifulSoup` (from `bs4`), `pandas` (as `pd`), `seaborn` (as `sns`) and any other packages you wish to use. 
* Set your Seaborn theme's style, context, and any runtime configurations you wish

In [1]:
#Import packages
import requests
import pandas as pd
from bs4 import BeautifulSoup
import seaborn as sns
import calendar

#Set Seaborn theme
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind', font_scale=0.8, rc={"axes.titlesize":20})

## 2. Scraping Water Supply Data 
We will be scraping data from the NC DEQs Local Water Supply Planning website, specifically the Durham's 2025 Municipal Local Water Supply Plan (LWSP): 
 * Navigate to https://www.ncwater.org/WUDC/app/LWSP/search.php
 * Set the year to 2025
 * Scroll down and select the LWSP link next to Durham Municipality. 
 * Note the web address: <https://www.ncwater.org/WUDC/app/LWSP/report.php?pwsid=03-32-010&year=2025>

---
### 2.1 Retrieve the website and parse it using BeautifulSoup

In [2]:
# 2.1 Retrieve the website and parse it using BeautifulSoup
website = 'https://www.ncwater.org/WUDC/app/LWSP/report.php?pwsid=03-32-010&year=2025'

response = requests.get(website)
response.raise_for_status()

web_text = BeautifulSoup(response.text, 'html.parser')

print(web_text.title.get_text(strip=True))

DWR :: Local Water Supply Planning


### 2.2 The data we want to collect are listed below:

* From the **1. System Information** section:
    - Water system name
    - Contact Person
    - Ownership
 
* From the **3. Water Supply Sources** section:
    - Maximum Day Use (MGD) - for each month

In the code chunk below scrape these values, assigning them to four separate variables.  
>HINT: The first value should be "`Durham`", the second "`Kieu Tran`", the third "`Municipality`", and the last should be a vector of 12 numeric values (represented as strings)".

In [3]:
# 2.2 Scrape the data into variables
water_sys_name = web_text.select_one('div+ table tr:nth-child(1) td:nth-child(2)').get_text(strip=True)

contact_person = web_text.select_one('div+ table tr:nth-child(4) td:nth-child(2)').get_text(strip=True)

ownership = web_text.select_one('div+ table tr:nth-child(2) td:nth-child(4)').get_text(strip=True)

max_day_use = web_text.select('th~ td+ td')
max_day_use = [node.get_text(strip=True) for node in max_day_use]

print(water_sys_name)
print(contact_person)
print(ownership)
print(max_day_use)

Durham
Kieu Tran
Municipality
['40.2400', '37.3800', '37.9100', '33.8100', '36.8400', '36.9000', '36.5300', '38.0400', '33.8000', '37.9500', '35.2000', '32.2000']


### 2.3 Construct a dataframe from the scraped data
Convert your scraped data into a dataframe. 
* This dataframe should have a column for each of the 4 variables scraped and a row for the month corresponding to the withdrawal data. 
* Note that you won't likely be able to scrape max water usage data in chronological order.
* Include columns for the month and year the data were collected. 
* Sort the data on year and month.
* Construct a datetime column from the year and month, assuming the day of the month will be the first. 

>🔥**TIP**: : To construct a list from a repeated set of values, use this approach:
>```python
>years_x6 = [2025] * 6
>print(years_x6)
>```

>🔥**TIP**: To construct a datetime column when you just the year and month, you can use the following:
>```python
>df['Date'] = pd.to_datetime({
>    "year": df['Year'],
>    "month": df['Month'],
>    "day": 1
>})
>```

In [4]:
# 2.3 Construct the dataframe from scraped data
month_order = [1,5,9,2,6,10,3,7,11,4,8,12]

df_durham_water = pd.DataFrame({
    "Month":month_order,
    "Year":2025,
    "Facility":water_sys_name,
    "Contact":contact_person,
    "Ownership":ownership,
    'Max_Day_Use':pd.to_numeric(max_day_use)
}).assign(
    Date = lambda df: pd.to_datetime({
        "year": df['Year'],
        "month":df["Month"],
        "day":1
    })
).sort_values(['Year','Month'])

df_durham_water.head()

,Month,Year,Facility,Contact,Ownership,Max_Day_Use,Date
0,1,2025,Durham,Kieu Tran,Municipality,40.24,2025-01-01
3,2,2025,Durham,Kieu Tran,Municipality,33.81,2025-02-01
6,3,2025,Durham,Kieu Tran,Municipality,36.53,2025-03-01
9,4,2025,Durham,Kieu Tran,Municipality,37.95,2025-04-01
1,5,2025,Durham,Kieu Tran,Municipality,37.38,2025-05-01


### 2.4 Plot the data
* Create a line plot of the maximum daily withdrawals across the months for 2024
* Label the x-axis with month names or abbreviated month names. (Note, you may have to do some more wrangling...)

In [5]:
# 2.4 Plot the data
df_durham_water['Month'] = pd.to_datetime(df_durham_water['Month'], format='%m')

df_durham_water['Month_Abb'] = pd.Categorical(
    values=df_durham_water['Month'].dt.month_name().str[:3],
    categories=calendar.month_abbr[1:]
)


In [12]:

sns.relplot(
    data=df_durham_water,
    kind='line',
    x='Month_Abb',
    y='Max_Day_Use'
).set(
    title='Max Day Use by Month, 2025 (MGD) ',
    xlabel="",
    ylabel='Withdrawal, MGD'
)

: 

## 3. Automating the Scraping Process
### 3.1 Construct the scraping function
Note that the PWSID and the year appear in the web address for the page we scraped. Construct a function with two inputs, "PWSID" and "year", that:
  - Creates a URL pointing to the LWSP for that PWSID for the given year
  - Creates a website object and scrapes the data from that object (just as you did above)
  - Constructs a dataframe from the scraped data, mostly as you did above, but includes the PWSID and year provided as function inputs in the dataframe. 
  - Returns the dataframe as the function's output

In [6]:
# 3.1 Create a scraping function
def scraper(PWSID, year):
    url = f'https://www.ncwater.org/WUDC/app/LWSP/report.php?pwsid={PWSID}&year={year}'
    response = requests.get(url)
    response.raise_for_status()
    web_text = BeautifulSoup(response.text, 'html.parser')

    water_sys_name = web_text.select_one('div+ table tr:nth-child(1) td:nth-child(2)').get_text(strip=True)
    contact_person = web_text.select_one('div+ table tr:nth-child(4) td:nth-child(2)').get_text(strip=True)
    ownership = web_text.select_one('div+ table tr:nth-child(2) td:nth-child(4)').get_text(strip=True)
    max_day_use = web_text.select('th~ td+ td')
    max_day_use = [node.get_text(strip=True) for node in max_day_use]

    df = pd.DataFrame({
        "PWSID":PWSID,
        "Month":month_order,
        "Year":year,
        "Facility":water_sys_name,
        "Contact":contact_person,
        "Ownership":ownership,
        'Max_Day_Use':pd.to_numeric(max_day_use)
        }).assign(
            Date = lambda df: pd.to_datetime({
            "year": df['Year'],
            "month":df["Month"],
            "day":1
            })
        ).sort_values(['Year','Month'])

    return df
    

### 3.2 Fetch and plot data for Durham (PWSID=`03-32-010`) in 2020
* Use the function above to extract and plot max daily withdrawals for Durham (PWSID=`03-32-010`) for each month in 2020.
* Reveal the structure of your dataframe using the `info()` function. 
* Then create plot your data in a line plot as above: Max Usage by Month.

In [8]:
# 3.2.1 Scrape data for 03-32-010 in 2020 & display its structure
df_durham_2020 = scraper('03-32-010', 2020)

df_durham_2020.info()

<class 'pandas.DataFrame'>
Index: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   PWSID        12 non-null     str           
 1   Month        12 non-null     int64         
 2   Year         12 non-null     int64         
 3   Facility     12 non-null     str           
 4   Contact      12 non-null     str           
 5   Ownership    12 non-null     str           
 6   Max_Day_Use  12 non-null     float64       
 7   Date         12 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 864.0 bytes


In [ ]:
# 3.2.2 Plot the data


### 3.3 Fetch and plot data for Cary (PWSID=`03-92-020`) in 2020

In [9]:
# 3.3.1 Fetch and plot data for Cary (PWSID=`03-92-020`) in 2020
df_cary_2020 = scraper('03-92-020', 2020)

df_cary_2020.info()

<class 'pandas.DataFrame'>
Index: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   PWSID        12 non-null     str           
 1   Month        12 non-null     int64         
 2   Year         12 non-null     int64         
 3   Facility     12 non-null     str           
 4   Contact      12 non-null     str           
 5   Ownership    12 non-null     str           
 6   Max_Day_Use  12 non-null     float64       
 7   Date         12 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 864.0 bytes


In [ ]:
# 3.3.2 Plot the Cary data


### 3.4 Examine monthy max discharge from 2020-2025
* Use list comprehension (or other methods) to produce a list of dataframes, one for each year in 2020 to 2025
* Concatenate these dataframes into a single one including data for all years
* Create a line plot of max monthly discharge across the full timespan (x-axis labels do not have to be months)

In [11]:
# 3.4.1 Generate a list of the dataframes
df_list = [scraper('03-32-010', the_year) for the_year in range(2020, 2026)]

# 3.4.2 Combine them into a single dataframe
df_durham_5yrs = pd.concat(df_list)

df_durham_5yrs.head()


,PWSID,Month,Year,Facility,Contact,Ownership,Max_Day_Use,Date
0,03-32-010,1,2020,Durham,Kieu Tran,Municipality,36.01,2020-01-01
3,03-32-010,2,2020,Durham,Kieu Tran,Municipality,32.05,2020-02-01
6,03-32-010,3,2020,Durham,Kieu Tran,Municipality,37.29,2020-03-01
9,03-32-010,4,2020,Durham,Kieu Tran,Municipality,32.37,2020-04-01
1,03-32-010,5,2020,Durham,Kieu Tran,Municipality,36.98,2020-05-01


In [ ]:
# 3.4.3 Plot
